# Silver: service ownership and quality

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"aidp_lab.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    target_location = location(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").option("path", target_location)
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
prepaid = spark.table(table("silver", "prepaid_service")).withColumn("address_id", F.lit(None).cast("string")).withColumn("technology", F.lit(None).cast("string"))
postpaid = spark.table(table("silver", "postpaid_service")).withColumn("address_id", F.lit(None).cast("string")).withColumn("technology", F.lit(None).cast("string"))
home = spark.table(table("silver", "home_service"))
columns = ["participant_key", "service_id", "service_number", "service_type", "customer_id", "product_id", "status", "monthly_value", "address_id", "technology"]
service_ownership = reduce(lambda left, right: left.unionByName(right), [prepaid.select(*columns), postpaid.select(*columns), home.select(*columns)])
write_delta(service_ownership, "silver", "service_ownership", "participant_key STRING, service_id STRING, service_number STRING, service_type STRING, customer_id STRING, product_id STRING, status STRING, monthly_value DECIMAL(14,2), address_id STRING, technology STRING")
quality = reduce(lambda left, right: left.unionByName(right), [
    spark.read.format("delta").load(location("silver", "_quality/customer")),
    spark.read.format("delta").load(location("silver", "_quality/prepaid")),
    spark.read.format("delta").load(location("silver", "_quality/postpaid")),
    spark.read.format("delta").load(location("silver", "_quality/home")),
])
write_delta(quality, "silver", "quality_issues", "participant_key STRING, dataset STRING, source_row_id STRING, record_key STRING, reason_code STRING, quarantined_at TIMESTAMP")
assert service_ownership.count() == 1261 and quality.count() == 30


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
